<a href="https://colab.research.google.com/github/cafekorea2000-prog/-healthcare/blob/main/gugak_stimuli_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎵 국악사전 1,674개 자극 일괄 생성

**박사 1년차 첫 학술대회 논문 (지하파라디오, 2026)**

## 📋 사용 방법 (위에서 아래로 차례대로)

1. **셀 1**: API 키 입력 (▶ 누르고 키 붙여넣기)
2. **셀 2**: 패키지 설치 (▶ 누르고 1분 기다림)
3. **셀 3**: 엑셀 파일 업로드 (▶ 누르고 파일 선택)
4. **셀 4**: ⭐ **시범 실행 5개** (▶ 누르고 1분 기다림 — 결과 미리 확인)
5. **셀 5**: 결과 미리보기 (▶ 누르고 5개 자극 점검)
6. **셀 6**: 본 실행 1,674개 (▶ 누르고 약 110분 기다림)
7. **셀 7**: 결과 다운로드 (▶ 누르면 자동 다운로드)

---

## ⚠️ 주의

- **셀 4 시범 실행**을 반드시 먼저 해보세요 (비용 약 $0.06)
- 시범 결과 *만족스러우면* 셀 6 본 실행 진행
- 셀 6 실행 중에는 **브라우저 탭 닫지 마세요**
- 도중에 끊겨도 다시 ▶ 누르면 *이어서 진행*됨

## 셀 1: API 키 안전 입력

▶ 버튼을 누르면 **입력창**이 뜹니다.

1. 발급받은 API 키 (sk-ant-api03-...) 복사
2. 입력창에 붙여넣기
3. 엔터

**⚠️ 키는 입력 후 *별표(****)로 가려져* 안전합니다.**

In [10]:
import getpass
import os

api_key = getpass.getpass('ANTHROPIC_API_KEY 입력 후 엔터: ')
os.environ['ANTHROPIC_API_KEY'] = api_key

if api_key.startswith('sk-ant-'):
    print('✅ 키 입력 완료')
else:
    print('⚠️ 키 형식이 sk-ant-로 시작하지 않음. 다시 확인하세요.')

ANTHROPIC_API_KEY 입력 후 엔터: ··········
✅ 키 입력 완료


## 셀 2: 패키지 설치 (1분 소요)

▶ 누르면 자동 설치

In [11]:
!pip install anthropic pandas openpyxl --quiet
print('✅ 패키지 설치 완료')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 10.0 MB/s eta 0:00:00
✅ 패키지 설치 완료


## 셀 3: 엑셀 파일 업로드

▶ 누르면 **파일 선택 버튼**이 뜹니다.

1. **국악_표제어_추출_1674.xlsx** 파일 선택
2. 업로드 완료 메시지 확인

In [12]:
from google.colab import files
import os

print('📂 "국악_표제어_추출_1674.xlsx" 파일을 선택하세요')
uploaded = files.upload()

# 업로드된 파일명 확인
for fname in uploaded.keys():
    if fname.endswith('.xlsx'):
        INPUT_FILE = '/content/' + fname
        print(f'\n✅ 업로드 완료: {fname}')
        break
else:
    print('❌ xlsx 파일이 업로드되지 않았습니다')

📂 "국악_표제어_추출_1674.xlsx" 파일을 선택하세요


Saving 국악_표제어_5카테고리_1674.xlsx to 국악_표제어_5카테고리_1674.xlsx

✅ 업로드 완료: 국악_표제어_5카테고리_1674.xlsx


## 셀 4: ⭐ 시범 실행 (5개만, 약 1분, 비용 약 $0.06)

**박사 1년차 첫 작업이니 반드시 시범 실행 먼저 해보세요.**

▶ 누르면 5개 표제어로 자극 생성하여 *형식과 품질 미리 확인* 가능

In [13]:
# ─────────────────────────────────────────────
# 자극 생성 함수 (V2.9.1 원칙 반영)
# ─────────────────────────────────────────────

import os
import json
import time
import pandas as pd
from anthropic import Anthropic
from pathlib import Path

client = Anthropic()  # 환경변수 ANTHROPIC_API_KEY 자동 사용

# Google Drive 마운트 (런타임 끊겨도 결과 보존)
from google.colab import drive
drive.mount('/content/drive')

MODEL = 'claude-sonnet-4-6'
OUTPUT_DIR = Path('/content/drive/MyDrive/stimuli_output')
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
RESULT_FILE = OUTPUT_DIR / 'stimuli_1674.jsonl'
ERROR_FILE = OUTPUT_DIR / 'errors.jsonl'

USEFUL_COLS = ['정의', '요약', '유래', '내용', '내용 및 구성', '의의 및 가치',
               '고문헌', '지정사항', '노랫말', '복식ㆍ의물ㆍ무구', '절차와 구성']

COL_MAX_LEN = {'정의': 300, '요약': 500, '유래': 800, '내용': 1500,
               '내용 및 구성': 1000, '의의 및 가치': 500, '고문헌': 200,
               '지정사항': 200, '노랫말': 300, '복식ㆍ의물ㆍ무구': 300, '절차와 구성': 500}

FEW_SHOT_EXAMPLES = '''다음은 5개 카테고리에서 1개씩 추출한 자극 작성 예시입니다.

## 예시 1 (종목·작품) — 황계사
출력: {"다차원": "한국 전통 성악곡의 한 갈래인 12가사 중 하나로(D5), 임에 대한 이별과 그리움을 표현한 노래(D4). 6박의 도드리장단으로 전체 8장이며 각 장에 규칙적으로 들어가는 \\"지화자 좋을시고\\"가 동일 선율로 반복된다(D3). 위 설명에 해당하는 한국 전통음악 용어는? [MASK]", "단일차원": "\\"지화자 좋을시고\\"가 반복되는 12가사 곡명은 무엇입니까? [MASK]", "다차원_차원": ["D3", "D4", "D5"], "단일차원_차원": ["D3"], "별칭_제외_여부": "별칭 부분 일치 없음"}

## 예시 2 (악기류) — 거문고
출력: {"다차원": "고구려에서 비롯되어 통일신라에 수용되었으며(D1), 8~9세기 옥보고가 새로운 음악을 지어 속명득에게 전한(D2) 한국 전통 현악기. 통일신라 대표 악기 삼현삼죽 중 하나(D3)로 정착하였고, 조선시대 선비들이 백악지장으로 숭상하며(D5) 술대로 줄을 쳐 연주하는 한국 전통 현악기는 무엇입니까? [MASK]", "단일차원": "옥보고가 이 악기로 음악을 지었다고 하는데, 해당 한국 전통 현악기는 무엇입니까? [MASK]", "다차원_차원": ["D1", "D2", "D3", "D4", "D5"], "단일차원_차원": ["D2"], "별칭_제외_여부": "별칭 부분 일치 없음"}

## 예시 3 (악기류 - 별칭 부분 일치 사례) — 아쟁
출력: {"다차원": "가야금처럼 옆으로 누운 긴 직육면체 울림통 위의 7현(또는 8~12현)을 활대로 문질러 소리 내는 현악기. 고려시대 당악기로 도입되어 조선시대에 향악기로 정착하였다. 위 설명에 해당하는 한국 전통음악 용어는? [MASK]", "단일차원": "직육면체 울림통 위에 7현을 얹고 활대로 문질러 소리 내는 한국 전통 현악기는 무엇입니까? [MASK]", "다차원_차원": ["D1", "D3", "D4", "D5"], "단일차원_차원": ["D3"], "별칭_제외_여부": "7현아쟁·정악아쟁 등 부분 일치 별칭 자극에서 제외"}

## 예시 4 (개념·이론) — 비가비
출력: {"다차원": "무계(巫系) 출신이 아닌 일반인 출신의 판소리 명창 또는 광대를 일컫는 용어. 정노식(鄭魯湜)이 1940년에 발간한 『조선창극사』에서 본격 서술되었으며, 결성 최선달(崔先達, 1726~1805), 권삼득, 정춘풍 등이 이에 해당한다고 기록되어 있다. 위 설명에 해당하는 한국 전통음악 용어는? [MASK]", "단일차원": "무계 출신이 아닌 일반인 판소리 명창을 일컫는 용어는 무엇입니까? [MASK]", "다차원_차원": ["D1", "D2", "D5"], "단일차원_차원": ["D5"], "별칭_제외_여부": "별칭 부분 일치 없음"}

## 예시 5 (기타) — 상쇠
출력: {"다차원": "꽹과리를 연주하며 전체 농악 연행을 지휘하고 이끌어 가는(D4) 농악패의 우두머리(D5). 행렬에서 가장 앞에 서서 부쇠·끝쇠를 이끄는 위치이다(D3). 위 설명에 해당하는 한국 전통음악 용어는? [MASK]", "단일차원": "꽹과리로 농악패 전체를 이끄는 우두머리를 가리키는 말은 무엇입니까? [MASK]", "다차원_차원": ["D3", "D4", "D5"], "단일차원_차원": ["D5", "D3"], "별칭_제외_여부": "별칭 부분 일치 없음. 본문에서 표제어 노출 차단"}
'''

SYSTEM_PROMPT = '''당신은 국악사전 표제어로부터 LLM 평가용 마스킹 자극을 생성하는 학술 도구입니다.

## 6대 정보 차원
- D1. 역사적 차원 (시대·기원·전승 경로)
- D2. 인물 차원 (창작자·연주자·전승자)
- D3. 형태 차원 (구조·재료·구성 요소)
- D4. 기능 차원 (용도·연주법·연행 맥락)
- D5. 위상 차원 (분류·평가·문화적 위치)
- D6. 어휘 차원 (별칭) — 자극 본문에는 활용 안 함, 채점에만 사용

## 다차원 조건 (Multi-cue)
- D1~D5 중 *3개 이상* 차원 활용, 약 150~200자

## 단일차원 조건 (Single-cue)
- D1~D5 중 *1차원* 원칙 (답 분산 위험 시 *최대 2차원*), 약 30~50자
- 우선순위: D2(인물) > D1(역사) > D5(위상) > D3(형태) > D4(기능)

## 절대 원칙
1. 표제어와 정확히 같은 글자가 부분적으로 포함된 별칭은 자극 본문에서 완전 제외
2. 자극 본문에 표제어가 *어떤 형태로도* 노출되면 절대 안 됨. 표제어가 짧거나 일반 단어와 겹치는 경우(예: "바디", "금", "아", "우", "응", "척" 등) 특히 주의하라. 표제어 직접 언급 대신 "이 학습 단위", "이 악기", "이 용어" 등 *우회적 지칭*을 사용하라
3. 자극은 반드시 다음 4가지 형식 중 하나로 끝나야 함:
   - "위 설명에 해당하는 한국 전통음악 용어는? [MASK]"
   - "~는 무엇입니까? [MASK]"
   - "~을/를 가리키는 말은 무엇입니까? [MASK]"
   - "~의 명칭은 무엇입니까? [MASK]"
   절대 금지: 동사로 끝나는 형태 (예: "지정된 [MASK]", "연주하는 [MASK]", "불리는 [MASK]"). 명사구 마지막에 무조건 "~는 무엇입니까?" 같은 의문문 어미를 붙여야 함.

## 출력 형식 (반드시 JSON, 다른 텍스트 절대 X)
{"다차원": "...", "단일차원": "...", "다차원_차원": ["D1", ...], "단일차원_차원": ["D2"], "별칭_제외_여부": "짧게"}'''


def build_user_prompt(row):
    parts = ['## 입력 데이터', f"- 표제어: {row['표제어']}"]
    if row.get('원어(Hanja)') and str(row['원어(Hanja)']) != 'nan':
        parts.append(f"- 원어(한자): {row['원어(Hanja)']}")
    if row.get('다른 이름') and str(row['다른 이름']) != 'nan':
        parts.append(f"- 별칭(다른 이름): {row['다른 이름']}")
    parts.append(f"- 카테고리: {row.get('카테고리', '?')}")
    for col in USEFUL_COLS:
        val = str(row.get(col, '')).strip()
        if val and val != 'nan':
            limit = COL_MAX_LEN.get(col, 500)
            if len(val) > limit:
                val = val[:limit] + ' ...'
            parts.append(f'- {col}: {val}')
    parts.append('')
    parts.append('위 데이터를 바탕으로 다차원·단일차원 자극을 JSON 형식으로 생성하세요.')
    return '\n'.join(parts)


def validate_output(output, target):
    errors = []
    for key in ['다차원', '단일차원', '다차원_차원', '단일차원_차원']:
        if key not in output:
            errors.append(f'키 누락: {key}')
    if errors:
        return False, '; '.join(errors)
    if target in output['다차원']:
        errors.append(f"다차원에 '{target}' 노출")
    if target in output['단일차원']:
        errors.append(f"단일차원에 '{target}' 노출")
    if '[MASK]' not in output['다차원']:
        errors.append('다차원 [MASK] 누락')
    if '[MASK]' not in output['단일차원']:
        errors.append('단일차원 [MASK] 누락')
    if len(output.get('다차원_차원', [])) < 3:
        errors.append(f"다차원 차원 부족: {len(output.get('다차원_차원', []))}")
    if len(output.get('단일차원_차원', [])) > 2:
        errors.append(f"단일차원 차원 초과: {len(output.get('단일차원_차원', []))}")
    return (False, '; '.join(errors)) if errors else (True, 'OK')


def generate_one(row, retry=0):
    target = row['표제어']
    user_prompt = FEW_SHOT_EXAMPLES + '\n\n' + build_user_prompt(row)
    try:
        response = client.messages.create(
            model=MODEL, max_tokens=1024, temperature=0.0,
            system=SYSTEM_PROMPT,
            messages=[{'role': 'user', 'content': user_prompt}],
        )
        text = response.content[0].text.strip()
        if text.startswith('```'):
            lines = text.split('\n')
            text = '\n'.join(lines[1:-1] if lines[-1].startswith('```') else lines[1:])
        if text.startswith('json'):
            text = text[4:].strip()
        output = json.loads(text)
        valid, msg = validate_output(output, target)
        if not valid:
            if retry < 3:
                time.sleep(3)
                return generate_one(row, retry + 1)
            print(f'  ❌ {target}: {msg}')
            with open(ERROR_FILE, 'a', encoding='utf-8') as f:
                f.write(json.dumps({'표제어': target, '오류': msg}, ensure_ascii=False) + '\n')
            return None
        output['표제어'] = target
        output['카테고리'] = row.get('카테고리', '')
        output['_usage'] = {'input': response.usage.input_tokens, 'output': response.usage.output_tokens}
        return output
    except Exception as e:
        if retry < 3:
            time.sleep(3 * (retry + 1))
            return generate_one(row, retry + 1)
        print(f'  ❌ {target}: {e}')
        with open(ERROR_FILE, 'a', encoding='utf-8') as f:
            f.write(json.dumps({'표제어': target, '오류': str(e)}, ensure_ascii=False) + '\n')
        return None


# ─────────────────────────────────────────────
# 시범 실행 — 5개
# ─────────────────────────────────────────────

print('📂 데이터 로드')
df = pd.read_excel(INPUT_FILE)

# 카테고리 컬럼 보강 (없으면 추가)
if '카테고리' not in df.columns:
    print('⚠️ 카테고리 컬럼 없음 → 분류 기반 자동 매핑')
    def map_cat(분류):
        s = str(분류)
        if '악기' in s or '의물' in s or '복식' in s or '무구' in s: return '악기류'
        if '문헌' in s or '악보' in s or '무보' in s or '시청각' in s: return '문헌류'
        if '종목' in s or '작품' in s: return '종목·작품'
        if '개념' in s or '이론' in s: return '개념·이론'
        return '기타'
    df['카테고리'] = df['분류'].apply(map_cat)

for col in USEFUL_COLS:
    if col in df.columns:
        df[col] = df[col].fillna('').astype(str)

print(f'   전체: {len(df)}개')
print(f'   카테고리 분포: {df["카테고리"].value_counts().to_dict()}')

# 5개 카테고리에서 1개씩 추출
import random
random.seed(42)
test_5 = []
for cat in ['종목·작품', '개념·이론', '악기류', '문헌류', '기타']:
    sub = df[df['카테고리'] == cat]
    if len(sub) > 0:
        test_5.append(sub.sample(1, random_state=42).iloc[0])

print(f'\n🧪 시범 실행 — 5개 표제어')
print('=' * 60)
test_results = []
for i, row in enumerate(test_5, 1):
    target = row['표제어']
    print(f'\n[{i}/5] {target} ({row["카테고리"]})', end=' ')
    result = generate_one(row.to_dict())
    if result:
        test_results.append(result)
        print(f'✅ (다차원 {len(result["다차원"])}자, 단일차원 {len(result["단일차원"])}자)')
    else:
        print('❌')

print(f'\n✅ 시범 실행 완료 — {len(test_results)}/5 성공')
print(f'   다음 셀에서 결과 미리보기')

Mounted at /content/drive
📂 데이터 로드
   전체: 1674개
   카테고리 분포: {'종목·작품': 871, '개념·이론': 291, '악기류': 218, '문헌류': 161, '기타': 133}

🧪 시범 실행 — 5개 표제어

[1/5] 서도잡가 (종목·작품) ✅ (다차원 186자, 단일차원 58자)

[2/5] 바디 (개념·이론) ✅ (다차원 172자, 단일차원 69자)

[3/5] 소고(벅구) (악기류) ✅ (다차원 196자, 단일차원 57자)

[4/5] 장자백 창본 춘향가 (문헌류) ✅ (다차원 206자, 단일차원 69자)

[5/5] 미지 (기타) ✅ (다차원 149자, 단일차원 53자)

✅ 시범 실행 완료 — 5/5 성공
   다음 셀에서 결과 미리보기


In [20]:
import os

# 이전 시범 5개 결과 삭제
result_path = '/content/stimuli_output/stimuli_1674.jsonl'
if os.path.exists(result_path):
    os.remove(result_path)
    print('✅ 이전 결과 파일 삭제됨')
else:
    print('ℹ️ 삭제할 파일 없음 (정상)')

# test_results 변수도 초기화
test_results = []
print('✅ test_results 초기화 완료')

ℹ️ 삭제할 파일 없음 (정상)
✅ test_results 초기화 완료


## 셀 5: 시범 결과 미리보기

5개 자극 *내용* 점검 — 만족스러우면 셀 6 진행

In [22]:
for i, r in enumerate(test_results, 1):
    print(f'\n{"="*70}')
    print(f'{i}. [{r["카테고리"]}] {r["표제어"]}')
    print(f'{"="*70}')
    print(f'\n📌 다차원 ({"+".join(r["다차원_차원"])} = {len(r["다차원_차원"])}차원, {len(r["다차원"])}자):')
    print(f'   {r["다차원"]}')
    print(f'\n📌 단일차원 ({"+".join(r["단일차원_차원"])} = {len(r["단일차원_차원"])}차원, {len(r["단일차원"])}자):')
    print(f'   {r["단일차원"]}')

print('\n' + '='*70)
print('📋 점검 사항:')
print('   1. 자극에 표제어가 노출되지 않는지')
print('   2. 다차원이 단일차원보다 정보 풍부한지')
print('   3. 자연스러운 한국어 문장인지')
print('   4. 모든 자극이 [MASK]로 끝나는지')
print('\n✅ 만족스러우면 → 셀 6 본 실행 진행')
print('❌ 문제 있으면 → 다시 답변 부탁드립니다')


📋 점검 사항:
   1. 자극에 표제어가 노출되지 않는지
   2. 다차원이 단일차원보다 정보 풍부한지
   3. 자연스러운 한국어 문장인지
   4. 모든 자극이 [MASK]로 끝나는지

✅ 만족스러우면 → 셀 6 본 실행 진행
❌ 문제 있으면 → 다시 답변 부탁드립니다


## 셀 6: 본 실행 — 1,674개 (약 110분, 비용 약 $20)

**⚠️ 시범 결과 만족스러운 경우에만 실행하세요.**

- 약 110분 소요
- 진행 상황 실시간 출력
- 도중에 끊겨도 다시 ▶ 누르면 *이어서* 진행
- **브라우저 탭 닫지 마세요**

In [25]:
# 시범 5개 결과를 본 실행 결과 파일에 우선 저장
with open(RESULT_FILE, 'w', encoding='utf-8') as f:
    for r in test_results:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

# 이미 처리된 표제어 확인
done = set()
if RESULT_FILE.exists():
    with open(RESULT_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                done.add(json.loads(line)['표제어'])
            except:
                pass
print(f'이미 완료: {len(done)}개')

todo = df[~df['표제어'].isin(done)].copy()
print(f'생성 필요: {len(todo)}개')

if len(todo) == 0:
    print('✅ 모든 표제어 이미 완료')
else:
    success = len(done)
    fail = 0
    total_in = 0
    total_out = 0

    for idx, (_, row) in enumerate(todo.iterrows(), 1):
        target = row['표제어']
        print(f'[{idx}/{len(todo)}] {target} ({row["카테고리"]})', end=' ')

        start = time.time()
        result = generate_one(row.to_dict())
        elapsed = time.time() - start

        if result:
            success += 1
            total_in += result['_usage']['input']
            total_out += result['_usage']['output']
            with open(RESULT_FILE, 'a', encoding='utf-8') as f:
                f.write(json.dumps(result, ensure_ascii=False) + '\n')
            print(f'✅ ({elapsed:.1f}s)')
        else:
            fail += 1
            print(f'❌ ({elapsed:.1f}s)')

        # 100개마다 중간 통계
        if idx % 100 == 0:
            cost = (total_in / 1_000_000) * 3.0 + (total_out / 1_000_000) * 15.0
            print(f'\n📊 중간 통계 ({idx}/{len(todo)}): 성공 {success}, 실패 {fail}, 비용 ${cost:.2f}\n')

    print('\n' + '='*60)
    print(f'✅ 본 실행 완료')
    print(f'   성공: {success}개')
    print(f'   실패: {fail}개')
    final_cost = (total_in / 1_000_000) * 3.0 + (total_out / 1_000_000) * 15.0
    print(f'   총 비용: ${final_cost:.2f} (약 {final_cost*1350:.0f}원)')

이미 완료: 0개
생성 필요: 1674개
[1/1674] Korean  Musical Instruments and an Introduction to Korean Music (문헌류) ✅ (7.9s)
[2/1674] 가곡선 (문헌류) ✅ (6.6s)
[3/1674] Koreanische Musik (문헌류) ✅ (6.6s)
[4/1674] 가 (악기류) 

KeyboardInterrupt: 

In [26]:
import os
import json

# 가능한 두 경로 모두 확인
paths = [
    '/content/drive/MyDrive/stimuli_output/stimuli_1674.jsonl',
    '/content/stimuli_output/stimuli_1674.jsonl',
]

for p in paths:
    if os.path.exists(p):
        with open(p, 'r', encoding='utf-8') as f:
            n = sum(1 for line in f if line.strip())
        print(f'✅ {p}')
        print(f'   → {n}개 자극 저장됨')
    else:
        print(f'❌ {p} (없음)')

# Drive 마운트 상태
print(f'\nDrive 마운트: {"됨" if os.path.exists("/content/drive/MyDrive") else "안 됨"}')

# 현재 RESULT_FILE 변수가 가리키는 경로
try:
    print(f'현재 RESULT_FILE: {RESULT_FILE}')
    print(f'   존재 여부: {os.path.exists(str(RESULT_FILE))}')
except NameError:
    print('RESULT_FILE 변수 없음 (셀 4 다시 실행 필요)')

✅ /content/drive/MyDrive/stimuli_output/stimuli_1674.jsonl
   → 3개 자극 저장됨
❌ /content/stimuli_output/stimuli_1674.jsonl (없음)

Drive 마운트: 됨
현재 RESULT_FILE: /content/drive/MyDrive/stimuli_output/stimuli_1674.jsonl
   존재 여부: True


## 셀 7: 결과 다운로드

▶ 누르면 자동으로 결과 파일이 PD님 컴퓨터로 다운로드됩니다.

In [24]:
from google.colab import files

# JSONL 다운로드
files.download(str(RESULT_FILE))

# 엑셀로 변환 (PD님이 보기 편하게)
import pandas as pd
results = []
with open(RESULT_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        results.append(json.loads(line))

df_out = pd.DataFrame([{
    '표제어': r['표제어'],
    '카테고리': r['카테고리'],
    '다차원_자극': r['다차원'],
    '다차원_차원수': len(r['다차원_차원']),
    '다차원_차원목록': '+'.join(r['다차원_차원']),
    '단일차원_자극': r['단일차원'],
    '단일차원_차원수': len(r['단일차원_차원']),
    '단일차원_차원목록': '+'.join(r['단일차원_차원']),
} for r in results])

EXCEL_FILE = '/content/stimuli_1674.xlsx'
df_out.to_excel(EXCEL_FILE, index=False)
files.download(EXCEL_FILE)

print(f'✅ 다운로드 완료')
print(f'   - stimuli_1674.jsonl ({len(results)}개 자극)')
print(f'   - stimuli_1674.xlsx (엑셀로도 변환)')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ 다운로드 완료
   - stimuli_1674.jsonl (1631개 자극)
   - stimuli_1674.xlsx (엑셀로도 변환)


In [3]:
import json
from pathlib import Path

RESULT_FILE = Path('/content/stimuli_output/stimuli_1674.jsonl')
ERROR_FILE = Path('/content/stimuli_output/errors.jsonl')

results = [] # Initialize results list
# 1) 결과 파일에 실제로 몇 개 들어있는지
if RESULT_FILE.exists(): # Add existence check
    with open(RESULT_FILE, 'r', encoding='utf-8') as f:
        results = [json.loads(line) for line in f if line.strip()]
    print(f'결과 파일 자극 수: {len(results)}')
else:
    print(f'경고: 결과 파일 {RESULT_FILE}이(가) 존재하지 않습니다. 이전 단계를 실행했는지 확인하세요.') # Inform user
    print('결과 파일 자극 수: 0')


# 2) 에러 파일도 확인
if ERROR_FILE.exists():
    with open(ERROR_FILE, 'r', encoding='utf-8') as f:
        errors = [json.loads(line) for line in f if line.strip()]
    print(f'에러 파일 항목 수: {len(errors)}')
else:
    print('에러 파일 없음')

# 3) 마지막 처리된 표제어
if results:
    print(f'마지막 표제어: {results[-1]["표제어"]}')
else:
    print('처리된 표제어 없음')

FileNotFoundError: [Errno 2] No such file or directory: '/content/stimuli_output/stimuli_1674.jsonl'

## 디렉토리 구조 확인

In [5]:
print('--- /content/ 디렉토리 내용 ---')
!ls -l /content/

print('\n--- /content/stimuli_output/ 디렉토리 내용 ---')
!ls -l /content/stimuli_output/

--- /content/ 디렉토리 내용 ---
total 4
drwxr-xr-x 1 root root 4096 Apr 16 13:28 sample_data

--- /content/stimuli_output/ 디렉토리 내용 ---
ls: cannot access '/content/stimuli_output/': No such file or directory


In [9]:
import os
print(os.path.exists('/content/stimuli_output'))
print(os.listdir('/content'))

False
['.config', 'sample_data']
